In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import random
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from tqdm import tqdm

### Download Mean_T1 data

In [ ]:
unet_myocardium_data = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update2/percentiles_T1.csv")
unet_septum_data = pd.read_csv("./data/shriya/SHMOLLI-output-unet-septum/mean_T1.csv")
VAE_myocardium_data = pd.read_csv("./data/shriya/latent_df_contrasted.txt",sep=' ')


In [ ]:
unet_septum_data = unet_septum_data.drop(columns=['Unnamed: 0'])


In [ ]:
unet_myocardium_data['Patient_ID'] = unet_myocardium_data['Patient_ID'].str[:7]
unet_myocardium_data = unet_myocardium_data.rename(columns={'Patient_ID': 'id'})

unet_myocardium_data

In [ ]:

unet_myocardium_data = unet_myocardium_data.sort_values(by='id')
unet_myocardium_data.reset_index(drop=True, inplace=True)

#unet_septum_data = unet_septum_data.sort_values(by='id')
#unet_septum_data.reset_index(drop=True, inplace=True)


In [ ]:
# Remove patients that the model was not able to segment

unet_myocardium_data = unet_myocardium_data[unet_myocardium_data["Mean_T1"] != 0]
#unet_septum_data = unet_septum_data[unet_septum_data["Mean_T1"] != 0]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

phenotype = "Mean_T1"

# Create histogram of values
plt.figure(figsize=(10, 6))
plt.hist(unet_myocardium_data[phenotype], bins=30, alpha=0.7, color='skyblue', edgecolor='black')

# Add labels and title
plt.xlabel(f'{phenotype} (ms)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title(f'Distribution of {phenotype} Values in Myocardium', fontsize=14, fontweight='bold')

### Download additional patient data

In [ ]:
data_1 = pd.read_csv("./data/bruna/lvedv/ukbb_all_no_exclusion_all_2_and_3_with_CMR_and_MR", delimiter='\t')

In [ ]:
normalized_LVM_data = pd.read_table("./data/bruna/nature_revision/features_nature.tsv.qnorm.txt")

### Verification of Masks

In [ ]:
import os
import pydicom
from PIL import Image
from tifffile import imread
import matplotlib.pyplot as plt
import cv2

In [ ]:
images_path = "./data/shriya/UKBB_SHMOLLI-pngimages"
masks_path = "./data/shriya/SHMOLLI-output-unet-myocardium-update2/SAM_masks"

image_files = [os.path.join(images_path, filename) for filename in sorted(os.listdir(images_path))]
mask_files = [os.path.join(masks_path, filename) for filename in sorted(os.listdir(masks_path))]

In [ ]:
def visualize_image_and_mask(index):
    image_path = image_files[index]
    mask_path = mask_files[index]

    image_array = np.array(Image.open(image_path))
    mask_array = np.array(Image.open(mask_path))

    # Ensure the mask is binary
    mask_array = np.where(mask_array > 0, 255, 0).astype(np.uint8)
    
    # Overlay the mask on top of the image
    overlay = cv2.bitwise_and(image_array, image_array, mask=mask_array)
    
    # Combine the image and overlay
    combined = cv2.addWeighted(image_array, 0, overlay, 1, 0)
    
    # Plot the combined image
    plt.imshow(combined)
    plt.axis('off')
    plt.show()

def visualize_image_and_mask_from_path(image_path, mask_path):
    image_array = np.array(Image.open(image_path))
    mask_array = np.array(Image.open(mask_path))

    # Ensure the mask is binary
    mask_array = np.where(mask_array > 0, 255, 0).astype(np.uint8)
    
    # Overlay the mask on top of the image
    overlay = cv2.bitwise_and(image_array, image_array, mask=mask_array)
    
    # Combine the image and overlay
    combined = cv2.addWeighted(image_array, 0, overlay, 1, 0)
    
    # Plot the original image and combined image side by side
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    # Original image
    axes[0].imshow(image_array, cmap='gray')
    axes[0].set_title("Original Image")
    axes[0].axis('off')
    
    # Combined image with overlay
    axes[1].imshow(combined, cmap='gray')
    axes[1].set_title("Image with Overlay")
    axes[1].axis('off')
    
    plt.show()

In [ ]:
def get_image_array(file_path):
    return np.array(Image.open(file_path))

In [ ]:
import random

idx = random.randint(0, len(image_files))

print(idx)
visualize_image_and_mask(idx)

In [ ]:
def check_completeness(contour):
    """Check if myocardial ring has large angular gaps"""
    if len(contour) < 10:
        return 0.0
    
    # Find center and calculate angles to all points
    M = cv2.moments(contour)
    if M["m00"] == 0:
        return 0.0
    
    center = np.array([M["m10"] / M["m00"], M["m01"] / M["m00"]])
    points = contour[:, 0, :]
    
    # Get angles from center to each point
    angles = [np.arctan2(p[1] - center[1], p[0] - center[0]) for p in points]
    angles = sorted([(a + 2*np.pi) % (2*np.pi) for a in angles])
    
    # Find largest angular gap
    gaps = [angles[i+1] - angles[i] for i in range(len(angles)-1)]
    gaps.append((2*np.pi - angles[-1]) + angles[0])  # wrap-around gap
    max_gap = max(gaps)
    
    # Complete rings shouldn't have gaps > 60 degrees
    return max(0, 1 - max(max_gap - np.pi/3, 0) / np.pi)

def check_quality(image_path):
    
    # Load the image (replace 'image_path' with your actual image path)
    image = np.array(Image.open(image_path))
    image = image[..., np.newaxis]
    
    # Apply binary thresholding to get a binary image
    binary_image = (image > 0).astype(np.uint8)
    
    # Find contours in the binary image
    contours, hierarchy = cv2.findContours(binary_image, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    
    # Initialize a flag to check for donut shape
    contains_donut = False

    #print(contours)
    
    # Create a copy of the original image to draw contours on
    image_with_contours = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    
    # Count to store the detected circles
    circle_count = 0
    
    # Loop through contours to check for circles
    for i, contour in enumerate(contours):
        if cv2.arcLength(contour, True) > 0:
            circle_count += 1
            
        cv2.drawContours(image_with_contours, [contour], -1, (0, 255, 0), 2)
    
    # Check if there are exactly two circles
    quality = (circle_count == 2)
    
    if (quality == False and contours):
        main_contour = max(contours, key=cv2.contourArea)
        completeness_score = check_completeness(main_contour)
        
        quality = (completeness_score > 0.98)
    
    return quality

In [ ]:
def find_mask_file_with_id(mask_files, id_value):
    """Find all files in mask_files list that contain the specified id"""
    return [file for file in mask_files if str(id_value) in file][0]

In [ ]:
for i in range(5):
    idx = random.randint(0, len(image_files))
    print("Good Quality?", check_quality(mask_files[idx]))
    visualize_image_and_mask(idx)

In [ ]:
unet_myocardium_data["quality_controlled"] = False

for index, row in tqdm(unet_myocardium_data.iterrows(), total=len(unet_myocardium_data), desc="Processing patients"):
    patient_id = row['id']
    mask_file = find_mask_file_with_id(mask_files, patient_id)
    quality = check_quality(mask_file)
        
    unet_myocardium_data.loc[index, "quality_controlled"] = quality

In [ ]:
clean_myocardium_data = unet_myocardium_data[unet_myocardium_data["quality_controlled"] == True]
clean_myocardium_data = clean_myocardium_data.drop('quality_controlled', axis=1)
clean_myocardium_data['id'] = clean_myocardium_data['id'].astype(str)

myocardium_csv_path = "./data/shriya/SHMOLLI-output-unet-myocardium-update2/cleaned_T1_percentiles.csv"
clean_myocardium_data.to_csv(myocardium_csv_path)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

clean_myocardium_data = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update2/cleaned_T1_percentiles.csv")

hypertension_patients = list(data_1[data_1["icd10"].astype(str).str.contains("I10", na=False)]["id"])
clean_myocardium_data['hypertension_status'] = clean_myocardium_data['id'].isin(hypertension_patients)

hematocrit_data = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium/hemocrit_only_trimmed.txt", sep='\t')

clean_myocardium_data['id'] = clean_myocardium_data['id'].astype('str')
hematocrit_data['IID'] = hematocrit_data['IID'].astype('str')

merged_data = pd.merge(clean_myocardium_data, hematocrit_data, left_on='id', right_on = "IID", how='inner')
merged_data = merged_data.drop(columns=['Unnamed: 0', 'FID', 'IID'])

# Define the T1 parameter columns
t1_columns = [
    'Mean_T1', 'T1_Standard_Deviation', 'T1_0.25th_Percentile', 
    'T1_1th_Percentile', 'T1_25th_Percentile', 'T1_50th_Percentile',
    'T1_75th_Percentile', 'T1_99th_Percentile', 'T1_99.75th_Percentile'
]

In [ ]:
def regress_out_covariates(df, outcome_cols, covariate_cols):
    """
    Regress out covariates from outcome variables and return residuals
    """
    regressed_data = df.copy()
    
    for outcome_col in outcome_cols:
        # Create feature matrix (covariates)
        X = df[covariate_cols].values
        y = df[outcome_col].values
        
        # Remove rows with missing values
        X_df = pd.DataFrame(X)
        y_series = pd.Series(y)

        mask = ~(X_df.isnull().any(axis=1) | y_series.isnull())
        X_clean = X[mask]
        y_clean = y[mask]
        
        if len(X_clean) > 0:
            # Fit linear regression
            reg = LinearRegression()
            reg.fit(X_clean, y_clean)
            
            # Calculate residuals for all data (predictions for missing values will be NaN)
            y_pred = np.full(len(df), np.nan)
            y_pred[mask] = reg.predict(X_clean)
            residuals = y - y_pred
            
            # Store residuals in new column
            new_col_name = f"{outcome_col}_regressed_{'_'.join(covariate_cols)}"
            regressed_data[new_col_name] = residuals
        else:
            # If no valid data, fill with NaN
            new_col_name = f"{outcome_col}_regressed_{'_'.join(covariate_cols)}"
            regressed_data[new_col_name] = np.nan
    
    return regressed_data

In [ ]:
hematocrit_col = 'hematocrit'  

print("Regressing out hematocrit...")
merged_data = regress_out_covariates(
    merged_data, 
    t1_columns, 
    [hematocrit_col]
)
print("Regressing out hematocrit and hypertension...")
merged_data = regress_out_covariates(
    merged_data, 
    t1_columns, 
    [hematocrit_col, 'hypertension_status']
)
hematocrit_regressed_cols = [col for col in merged_data.columns if '_regressed_hematocrit' in col]
both_regressed_cols = [col for col in merged_data.columns if '_regressed_hematocrit_hypertension' in col]

In [ ]:
regressed_csv_path = "./data/shriya/SHMOLLI-output-unet-myocardium-update2/cleaned_T1_percentiles_HHregressed.csv"
merged_data.to_csv(regressed_csv_path)

In [ ]:
clean_septum_data = unet_septum_data[unet_septum_data["quality_controlled"] == True]

clean_myocardium_data = clean_myocardium_data.rename(columns={'Mean_T1': 'Mean_T1_myocardium', 
                                                              'T1_50th_Percentile': 'T1_50th_Percentile_myocardium', 
                                                              'T1_Standard_Deviation': 'T1_Standard_Deviation_myocardium', 
                                                              'T1_99th_Percentile': 'T1_99th_Percentile_myocardium', 
                                                              'T1_75th_Percentile': 'T1_75th_Percentile_myocardium'})

clean_myocardium_data = clean_myocardium_data.drop('quality_controlled', axis=1)
clean_septum_data = clean_septum_data.drop('quality_controlled', axis=1)
clean_septum_data = clean_septum_data.drop('HCM_status', axis=1)

# Convert 'id' columns to the same type
data_1['id'] = data_1['id'].astype(str)
clean_myocardium_data['id'] = clean_myocardium_data['id'].astype(str)
clean_septum_data['id'] = clean_septum_data['id'].astype(str)

merged_clean_myocardium_data = pd.merge(clean_myocardium_data, data_1, on='id')
merged_clean_septum_data = pd.merge(clean_septum_data, data_1, on='id')

merged_clean_all_data = pd.merge(clean_septum_data, merged_clean_myocardium_data, on='id')

In [ ]:
myocardium_csv_path = "./data/shriya/SHMOLLI-output-unet-myocardium/cleaned_mean_T1_allpheno.csv"
septum_csv_path = "./data/shriya/SHMOLLI-output-unet-septum/cleaned_mean_T1_allpheno.csv"

merged_clean_all_data.to_csv(myocardium_csv_path, index=False)
merged_clean_all_data.to_csv(septum_csv_path, index=False)

In [ ]:
# List of variables to test
variables = ['Mean_T1', 'T1_50th_Percentile', 'T1_Standard_Deviation', 'T1_99th_Percentile', 'T1_75th_Percentile']

# Function to perform t-test
def perform_t_test(data, variable, group_col):
    group1 = data[data[group_col] == True][variable]
    group2 = data[data[group_col] == False][variable]
    mean_group1 = group1.mean()
    mean_group2 = group2.mean()
    t_stat, p_value = stats.ttest_ind(group1, group2, nan_policy='omit')
    return t_stat, p_value, mean_group1, mean_group2

# Perform t-tests and print results
for var in variables:
    t_stat, p_value, mean_group1, mean_group2 = perform_t_test(clean_septum_data, var, 'HCM_status')
    print(f"{var}: t-statistic = {t_stat}, p-value = {p_value}")
    print(f"Mean of group 1 (HCM_status=True): {mean_group1}")
    print(f"Mean of group 2 (HCM_status=False): {mean_group2}")

### Add HCM Status to Mean_T1 data

In [ ]:
# Get the list of all Patient IDs with HCM diagnosis

HCM_patients = data_1[data_1['icd10'].str.contains(r'I421|I422', na=False)]
DCM_patients = data_1[data_1['icd10'].str.contains(r'I420', na=False)]
valvular_patients = data_1[data_1['icd10'].str.contains(r'I509|I089|I359', na=False)] #I509: Heart failure, I089: rheumatic multiple valve disease, I359: Nonrheumatic aortic valve disorder
amyloidosis_patients = data_1[data_1['icd10'].str.contains(r'E85', na=False)]
restrictiveCM_patients = data_1[data_1['icd10'].str.contains(r'I425', na=False)]

ischemic_patients = data_1[data_1['icd10'].str.contains(r'I20|I21|I22|I23|I24', na=False)]
nonischemic_patients = data_1[data_1['icd10'].str.contains(r'I42|I50|I31|I34|I35', na=False)]
quality_patients = unet_septum_data[unet_septum_data['quality_controlled'] == True]

HCM_patients_ids = (HCM_patients["id"].astype(str).tolist())
DCM_patients_ids = (DCM_patients["id"].astype(str).tolist())
valvular_patients_ids = (valvular_patients["id"].astype(str).tolist())
amyloidosis_patients_ids = (amyloidosis_patients["id"].astype(str).tolist())
restrictiveCM_patients_ids = (restrictiveCM_patients["id"].astype(str).tolist())

ischemic_patients_ids = (ischemic_patients["id"].astype(str).tolist())
nonischemic_patients_ids = (nonischemic_patients["id"].astype(str).tolist())
quality_patients_ids = (quality_patients["id"].astype(str).tolist())


In [ ]:
CAD_patients = data_1[data_1['icd10'].str.contains(r'I20|I21|I22|I23|I24', na=False)]
CAD_graft_patients = data_1[data_1['icd10'].str.contains(r'I42|I50|I31|I34|I35', na=False)]
valvular_patients = data_1[data_1['icd10'].str.contains(r'I509|I089|I359', na=False)] #I509: Heart failure, I089: rheumatic multiple valve disease, I359: Nonrheumatic aortic valve disorder
heart_failure_patients = data_1[data_1['icd10'].str.contains(r'I509|I089|I359', na=False)] #I509: Heart failure, I089: rheumatic multiple valve disease, I359: Nonrheumatic aortic valve disorder
cardiac_arrest_patients = data_1[data_1['icd10'].str.contains(r'I509|I089|I359', na=False)] #I509: Heart failure, I089: rheumatic multiple valve disease, I359: Nonrheumatic aortic valve disorder
MI_patients = data_1[data_1['icd10'].str.contains(r'I509|I089|I359', na=False)] #I509: Heart failure, I089: rheumatic multiple valve disease, I359: Nonrheumatic aortic valve disorder

quality_patients = unet_septum_data[unet_septum_data['quality_controlled'] == True]

In [ ]:
unet_myocardium_data['HCM_status'] = unet_myocardium_data['id'].isin(HCM_patients_ids)
unet_septum_data['HCM_status'] = unet_septum_data['id'].isin(HCM_patients_ids)

unet_myocardium_data['DCM_status'] = unet_myocardium_data['id'].isin(HCM_patients_ids)
unet_septum_data['DCM_status'] = unet_septum_data['id'].isin(HCM_patients_ids)

unet_myocardium_data['valvular_status'] = unet_myocardium_data['id'].isin(valvular_patients_ids)
unet_septum_data['valvular_status'] = unet_septum_data['id'].isin(valvular_patients_ids)

unet_myocardium_data['amyloidosis_status'] = unet_myocardium_data['id'].isin(amyloidosis_patients_ids)
unet_septum_data['amyloidosis_status'] = unet_septum_data['id'].isin(amyloidosis_patients_ids)

unet_myocardium_data['restrictiveCM_status'] = unet_myocardium_data['id'].isin(restrictiveCM_patients_ids)
unet_septum_data['restrictiveCM_status'] = unet_septum_data['id'].isin(restrictiveCM_patients_ids)

unet_myocardium_data['ischemic_disease'] = unet_myocardium_data['id'].isin(ischemic_patients_ids)
unet_septum_data['ischemic_disease'] = unet_septum_data['id'].isin(ischemic_patients_ids)

unet_myocardium_data['nonischemic_disease'] = unet_myocardium_data['id'].isin(nonischemic_patients_ids)
unet_septum_data['nonischemic_disease'] = unet_septum_data['id'].isin(nonischemic_patients_ids)

unet_myocardium_data['quality_controlled'] = unet_myocardium_data['id'].isin(quality_patients_ids)
unet_septum_data['quality_controlled'] = unet_septum_data['id'].isin(quality_patients_ids) # Sets everything to false?


In [ ]:
HCM_myocardium_data = unet_myocardium_data[(unet_myocardium_data['HCM_status'] == True) & (unet_myocardium_data['quality_controlled'] == True)]
DCM_myocardium_data = unet_myocardium_data[(unet_myocardium_data['DCM_status'] == True) & (unet_myocardium_data['quality_controlled'] == True)]
valvular_myocardium_data = unet_myocardium_data[(unet_myocardium_data['valvular_status'] == True) & (unet_myocardium_data['quality_controlled'] == True)]
amyloidosis_myocardium_data = unet_myocardium_data[(unet_myocardium_data['amyloidosis_status'] == True) & (unet_myocardium_data['quality_controlled'] == True)]
restrictiveCM_myocardium_data = unet_myocardium_data[(unet_myocardium_data['restrictiveCM_status'] == True) & (unet_myocardium_data['quality_controlled'] == True)]

nonischemic_myocardium_data = unet_myocardium_data[(unet_myocardium_data['ischemic_disease'] == True) & (unet_myocardium_data['quality_controlled'] == True)]
ischemic_myocardium_data = unet_myocardium_data[(unet_myocardium_data['nonischemic_disease'] == True) & (unet_myocardium_data['quality_controlled'] == True)]
normal_myocardium_data = unet_myocardium_data[(unet_myocardium_data['nonischemic_disease'] == False) & (unet_myocardium_data['ischemic_disease'] == False) & (unet_myocardium_data['quality_controlled'] == True)]

In [ ]:
var = 'Mean_T1'

table = unet_myocardium_data[unet_myocardium_data["ischemic_disease"] == True]

# Plot the distributions
plt.figure(figsize=(10, 6))

# Histogram for list1
plt.hist(table[var], bins=1000, alpha=0.5, color='blue', edgecolor='black')

# Adding labels and title
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.title(var)
plt.legend(loc='upper right')

# Show plot
plt.show()

In [ ]:
unet_myocardium_data.to_csv("./data/shriya/SHMOLLI-output-unet-myocardium/mean_T1.csv")
unet_septum_data.to_csv("./data/shriya/SHMOLLI-output-unet-septum/mean_T1.csv")


In [ ]:
# Filter to get IDs of normal patients
unet_septum_ids = [id for id, qc in zip(unet_septum_data['id'], unet_septum_data['quality_controlled']) if qc]

# Create sets for easier manipulation
quality_patients_set = set(quality_patients_ids)
HCM_set = set(HCM_patients_ids)
DCM_set = set(DCM_patients_ids)
valvular_set = set(valvular_patients_ids)
amyloidosis_set = set(amyloidosis_patients_ids)
restrictiveCM_set = set(restrictiveCM_patients_ids)
ischemic_set = set(ischemic_patients_ids)
nonischemic_set = set(nonischemic_patients_ids)

In [ ]:
# Create dictionary to store results
results = {}

# Function to find overlapping IDs and count
def find_overlaps(category_set, category_name):
    overlaps = category_set.intersection(quality_patients_set)
    results[category_name] = {
        'ids': list(overlaps),
        'count': len(overlaps)
    }

In [ ]:
# Check each category
find_overlaps(HCM_set, 'HCM')
find_overlaps(DCM_set, 'DCM')
find_overlaps(valvular_set, 'Valvular')
find_overlaps(amyloidosis_set, 'Amyloidosis')
find_overlaps(restrictiveCM_set, 'Restrictive CM')
find_overlaps(ischemic_set, 'Ischemic')
find_overlaps(nonischemic_set, 'Non-Ischemic')


In [ ]:
# Find IDs in unet_septum_data that do not overlap with any of the other lists
all_overlaps = HCM_set.union(DCM_set, valvular_set, amyloidosis_set, restrictiveCM_set, ischemic_set, nonischemic_set)
normal_patients_set = set(unet_septum_ids) - all_overlaps
results['Normal Patients'] = {
    'ids': list(normal_patients_set),
    'count': len(normal_patients_set)
}


In [ ]:
# Write results to a text file
with open("./data/shriya/SHMOLLI-output-unet-myocardium/disease_patient_IDs.txt", 'w') as file:
    for category, data in results.items():
        file.write(f"{category} IDs: {', '.join(map(str, data['ids']))}\n")
        file.write(f"Number of IDs in {category}: {data['count']}\n\n")

print("Summary written to patient_ids_summary.txt")

In [ ]:
for category, data in results.items():
    print(f"Number of IDs in {category}: {data['count']}")

### Quality Control Data

### Prepare Covarities Table

In [ ]:
#phenotypes = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium/cleaned_mean_T1_allpheno.csv", low_memory=False)
covariates = pd.read_csv("./data/ukbb_code/anna_code/preprocessing/covariates.txt", delimiter='\t')

In [ ]:
def prepare_covarites(phenotypes, covariates, save_name):
    phenotypes_no_duplicates = phenotypes.drop_duplicates(subset=['IID'], keep='first')
    phenotypes_no_duplicates['IID'] = phenotypes_no_duplicates['IID'].astype(int)
    
    # Perform an inner join on 'id' and 'IID' to ensure both DataFrames have the exact same rows
    merged_df = pd.merge(phenotypes_no_duplicates, covariates, on='IID')
    merged_df.replace(-1000, np.nan, inplace=True)
    merged_df = merged_df.dropna()

    print(merged_df)
    
    # Separate the merged DataFrame into trimmed DataFrames
    phenotypes_trimmed = merged_df[['FID', 
        'IID',
        '0', 
        '1',
        '2', 
        '3',  
        '4', 
        '5']]
    covariates_trimmed = merged_df[list(covariates.columns)]

    # Save phenotypes_trimmed to a CSV file
    phenotypes_trimmed.to_csv(f'./data/shriya/SHMOLLI-VAE-output/{save_name}_phenotypes_trimmed.txt', sep='\t', index=False)
    
    # Save covariates_trimmed to a CSV file
    covariates_trimmed.to_csv(f'./data/shriya/SHMOLLI-VAE-output/{save_name}_covariates_trimmed.txt', sep='\t', index=False)

    return phenotypes_trimmed, covariates_trimmed

In [ ]:
#clean_VAE_myocardium_data = VAE_myocardium_data[unet_myocardium_data["quality_controlled"] == True]
VAE_myocardium_data
phenotypes_trimmed, covariates_trimmed = prepare_covarites(VAE_myocardium_data, covariates, "VAE_dimensions")

In [ ]:
# phenotypes_trimmed.drop(columns=["FID", "IID"], inplace=True)
# covariates_trimmed.drop(columns=["FID", "IID"], inplace=True)

# Initialize a list to store results
results = []

# Loop through each pair of columns
for phenotype in phenotypes_trimmed.columns:
    for covariate in covariates_trimmed.columns:
        # Compute the Pearson correlation and p-value
        corr, p_value = pearsonr(phenotypes_trimmed[phenotype], covariates_trimmed[covariate])
        
        # Append the result as a dictionary
        results.append({
            "Phenotype": phenotype,
            "Covariate": covariate,
            "Correlation": corr,
            "P-Value": p_value
        })

# Convert results to a DataFrame
results_df = pd.DataFrame(results)

# Sort results by absolute correlation value (optional)
results_df = results_df.sort_values(by="Correlation", key=abs, ascending=False)

# Print the results
for _, row in results_df.iterrows():
    if row["P-Value"] < 0.1:
        print(row["Phenotype"], row["Covariate"]) # row["P-Value"],

In [ ]:
prepare_covarites(HCM_myocardium_data, covariates, "HCM")
prepare_covarites(DCM_myocardium_data, covariates, "DCM")
prepare_covarites(valvular_myocardium_data, covariates, "valvular")
prepare_covarites(amyloidosis_myocardium_data, covariates, "amyloidosis")
prepare_covarites(restrictiveCM_myocardium_data, covariates, "restrictiveCM")
prepare_covarites(nonischemic_myocardium_data, covariates, "nonischemic")
prepare_covarites(ischemic_myocardium_data, covariates, "ischemic")
prepare_covarites(normal_myocardium_data, covariates, "normal")

In [ ]:
#Transform files into Manul Riva IDs
#need to make sure patient mapping cell is run first

def transform_residuals_IDs(save_name):
    phenotypes = f'./data/shriya/SHMOLLI-VAE-output/{save_name}_phenotypes.no_outliers.residuals.qnorm.txt'
    imputed_phenotypes = f'./data/shriya/SHMOLLI-VAE-output/{save_name}_phenotypes_imputed.no_outliers.residuals.qnorm.txt'
    
    imputed_residuals = pd.read_csv(phenotypes, sep='\t') 
    print(imputed_residuals)
    
    for index, row in imputed_residuals.iterrows():
        IID = row["IID"]
        mapped_IID = reverse_mapped_ID(int(IID), patient_mapping)
    
        imputed_residuals.at[index, "IID"] = mapped_IID
        imputed_residuals.at[index, "FID"] = mapped_IID
    
    print(imputed_residuals)
    
    imputed_residuals = imputed_residuals.dropna()
    
    imputed_residuals.to_csv(imputed_phenotypes, sep='\t', index=False)

In [ ]:
transform_residuals_IDs("VAE_dimensions")

In [ ]:
transform_residuals_IDs("nonischemic")
transform_residuals_IDs("normal")

### Legacy Code

In [ ]:
imputed_residuals = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium/T1_phenotypes.no_outliers.residuals.qnorm.txt", sep='\t') 

In [ ]:
for index, row in imputed_residuals.iterrows():
    IID = row["IID"]
    mapped_IID = reverse_mapped_ID(int(IID), patient_mapping)

    imputed_residuals.at[index, "IID"] = mapped_IID
    imputed_residuals.at[index, "FID"] = mapped_IID

In [ ]:
imputed_residuals = imputed_residuals.dropna(subset=['IID'])
imputed_residuals['IID'] = imputed_residuals['IID'].astype(int)
imputed_residuals['FID'] = imputed_residuals['FID'].astype(int)

In [ ]:
imputed_residuals = imputed_residuals.dropna()

In [ ]:
imputed_residuals.to_csv('./data/shriya/SHMOLLI-output-unet-myocardium/T1_phenotypes_imputed.no_outliers.residuals.qnorm.txt', sep='\t', index=False)

In [ ]:
european_ancestry = pd.read_csv("./data/bruna/euro_minus_first/euro_minus_exclusion_minus_firstdegree.txt", sep='\t', header = None) 

In [ ]:
european_ancestry = european_ancestry.dropna()
european_ancestry

In [ ]:
for index, row in european_ancestry.iterrows():
    IID = row[0]
    mapped_IID = reverse_mapped_ID(int(IID), patient_mapping)

    european_ancestry.at[index, 0] = mapped_IID
    european_ancestry.at[index, 1] = mapped_IID

In [ ]:
european_ancestry.to_csv("./data/bruna/euro_minus_first/euro_minus_exclusion_minus_firstdegree_imputed.txt", sep='\t', index=False)

### Check Associations with Normalized LVM Status

In [ ]:
normalized_LVM_data["iLVM"].replace(-1000, np.nan, inplace=True)

In [ ]:
normalized_LVM_data.dropna(inplace=True)

In [ ]:
patient_mapping = pd.read_table("./data/shriya/ukb22282_24983_mapping.tsv", header = None)

# Function to get the value from column 0 based on the ID in column 1
def get_mapped_ID(id_to_find, mapping_df):
    # Check if the ID exists in column 1
    if id_to_find in mapping_df[1].values:
        # Find the index of the row where the ID is located
        row_index = mapping_df[mapping_df[1] == id_to_find].index[0]
        # Return the value from column 0 in the same row
        return mapping_df.loc[row_index, 0]
    else:
        return None  # Return None if the ID is not found

# Function to get the value from column 0 based on the ID in column 1
def reverse_mapped_ID(id_to_find, mapping_df):
    # Check if the ID exists in column 1
    if id_to_find in mapping_df[0].values:
        # Find the index of the row where the ID is located
        row_index = mapping_df[mapping_df[0] == id_to_find].index[0]
        # Return the value from column 0 in the same row
        return int(mapping_df.loc[row_index, 1])
    else:
        return None  # Return None if the ID is not found

In [ ]:
iLVM_99 = normalized_LVM_data['iLVM'].quantile(0.99)
subset_iLVM_99 = normalized_LVM_data[normalized_LVM_data['iLVM'] >= iLVM_99]
iLVM_99_ids = subset_iLVM_99['IID'].tolist()
iLVM_99_ids = [get_mapped_ID(id, patient_mapping) for id in iLVM_99_ids]

iLVM_1 = normalized_LVM_data['iLVM'].quantile(0.01)
subset_iLVM_1 = normalized_LVM_data[normalized_LVM_data['iLVM'] <= iLVM_1]
iLVM_1_ids = subset_iLVM_1['IID'].tolist()
iLVM_1_ids = [get_mapped_ID(id, patient_mapping) for id in iLVM_1_ids]

### Check Myocardium vs Septum T1 Value Similarity 

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
phenotypes = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium/cleaned_mean_T1_allpheno.csv", low_memory=False)
regressed_phenotypes = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium/T1_phenotypes.no_outliers.residuals.qnorm.txt", sep='\t') 
latent_dimensions = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update2/cleaned_latent_dimentions_PHEWAS.no_outliers.residuals.qnorm.txt", sep='\t')

In [ ]:
septal_T1 = pd.read_csv("./data/shriya/SHMOLLI-output-unet-septum/mean_T1.csv")
septal_T1

In [ ]:
unfiltered_unet_t1_raw = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update2/percentiles_T1.csv")
unfiltered_unet_t1_raw["Patient_ID"] = [int(x[:7]) for x in unfiltered_unet_t1_raw["Patient_ID"]]

genetic_filtered_unet_t1_raw = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update2/cleaned_T1_percentiles_HHregressed.csv_phenotypes_trimmed.txt", sep='\t')

valid_images = np.load("./data/shriya/SHMOLLI-output-unet-myocardium-update2/quality_ID_list.csv.npy").tolist()
valid_images = [int(x) for x in valid_images]

In [ ]:
from scipy import stats
from scipy.stats import ttest_rel

def bland_altman_plot(ground_truth, predicted):

    ground_truth = np.array(ground_truth)
    predicted = np.array(predicted)

    mean = np.mean([ground_truth, predicted], axis=0)
    diff = ground_truth - predicted
    mean_diff = np.mean(diff)
    std_diff = np.std(diff, axis=0)
    limits_of_agreement = 1.96 * std_diff

    t_statistic, p_value = ttest_rel(ground_truth, predicted)

    plt.figure(figsize=(8, 6))
    plt.scatter(mean, diff, color='blue', alpha=0.5)
    plt.axhline(mean_diff, color='gray', linestyle='--')
    plt.axhline(mean_diff + limits_of_agreement, color='red', linestyle='--')
    plt.axhline(mean_diff - limits_of_agreement, color='red', linestyle='--')
    plt.xlabel('Mean of Myocardium and Septum')
    plt.ylabel('Difference (Septum - Myocardium)')
    plt.title('Bland-Altman Plot')
    plt.grid(True)
    plt.show()

    return mean_diff, limits_of_agreement, std_diff, t_statistic, p_value

def perform_t_test(list1, list2):
    group1 = list1
    group2 = list2
    mean_group1 = group1.mean()
    mean_group2 = group2.mean()
    t_stat, p_value = stats.ttest_ind(group1, group2, nan_policy='omit')
    return t_stat, p_value, mean_group1, mean_group2

def get_stats(list1, list2):
    mean_diff, limits_of_agreement, std_diff, t_statistic, p_value = bland_altman_plot(list1, list2)
    print("Mean Difference:", mean_diff)
    print("Limits of Agreement:", limits_of_agreement)
    print("Standard Deviation of Differences:", std_diff)
    print("t-statistic:", t_statistic)
    print("p-value:", p_value)
    # Perform linear regression
    slope, intercept, r_value, p_value, std_err = stats.linregress(list1, list2)
    
    # Print the slope, intercept, and p-value
    print(f"Slope: {slope}")
    print(f"Intercept: {intercept}")
    print(f"P-value: {p_value}")
    
    # Generate data points for best fit line
    x = [i for i in range(700, 1200)]  # Choose x values from -10 to 10
    y = slope * np.array(x) + intercept  # Since x = y, y values are same as x values
    
    
    # Plot data and best fit line
    plt.scatter(list1, list2, label='Data Points')
    plt.plot(x, y, color='blue', label='Best Fit Line')
    plt.xlabel("Septum")
    plt.ylabel("Myocardium")
    plt.title('Best Fit Line')
    plt.legend()

    print("T-Test results")
    t_stat, p_value, mean_group1, mean_group2 = perform_t_test(list1, list2)
    print(f"t-statistic = {t_stat}, p-value = {p_value}")
    print(f"Mean of group 1 (Septum): {mean_group1}")
    print(f"Mean of group 2 (Myocardium): {mean_group2}")

In [ ]:
septum_var = np.array(phenotypes["T1_Standard_Deviation_septum"])
myocardium_var = np.array(phenotypes["T1_Standard_Deviation_myocardium"])

get_stats(septum_var, myocardium_var)

In [ ]:
septum_var = np.array(phenotypes["Mean_T1_septum"])
myocardium_var = np.array(phenotypes["Mean_T1_myocardium"])

get_stats(septum_var, myocardium_var)

In [ ]:
septum_var = np.array(phenotypes["Mean_T1_septum"])
myocardium_var = np.array(phenotypes["T1_75th_Percentile_myocardium"])

get_stats(septum_var, myocardium_var)

In [ ]:
from scipy.stats import spearmanr
import pandas as pd
import numpy as np

# Filter to valid images
phenotypes_valid = septal_T1[septal_T1["id"].isin(valid_images)]
unfiltered_valid = unfiltered_unet_t1_raw[unfiltered_unet_t1_raw["Patient_ID"].isin(valid_images)]
latent_valid = latent_dimensions[latent_dimensions["IID"].isin(valid_images)]

# Target variable
target = phenotypes_valid[["id", "Mean_T1_septum"]].dropna().rename(columns={"id": "Patient_ID"})

results = []

for df_name, df, id_col in [
    ("unfiltered_unet_t1_raw", unfiltered_valid, "Patient_ID"),
    ("latent_dimensions",      latent_valid,      "IID"),
]:
    df_renamed = df.rename(columns={id_col: "Patient_ID"})
    merged = target.merge(df_renamed, on="Patient_ID", how="inner")
    feature_cols = [c for c in df_renamed.columns if c != "Patient_ID"]

    for col in feature_cols:
        sub = merged[["Mean_T1_septum", col]].dropna()
        if len(sub) < 10:
            continue
        r, p = spearmanr(sub["Mean_T1_septum"], sub[col])
        results.append({"source": df_name, "feature": col, "spearman_r": r, "p_value": p})

results_df = pd.DataFrame(results).sort_values("p_value")
results_df["significant"] = results_df["p_value"] < 0.05

print(f"Total comparisons: {len(results_df)}")
print(f"Significant (p<0.05): {results_df['significant'].sum()}")
print("\nTop 20 by |r|:")
print(results_df.reindex(results_df["spearman_r"].abs().sort_values(ascending=False).index).head(20).to_string(index=False))

In [ ]:
from scipy.stats import pearsonr
import pandas as pd
import numpy as np

# Filter to valid images
phenotypes_valid = phenotypes[phenotypes["id"].isin(valid_images)]
unfiltered_valid = unfiltered_unet_t1_raw[unfiltered_unet_t1_raw["Patient_ID"].isin(valid_images)]
latent_valid = latent_dimensions[latent_dimensions["IID"].isin(valid_images)]

# Target variable
target = phenotypes_valid[["id", "Mean_T1_septum"]].dropna().rename(columns={"id": "Patient_ID"})

results = []

for df_name, df, id_col in [
    ("unfiltered_unet_t1_raw", unfiltered_valid, "Patient_ID"),
    ("latent_dimensions",      latent_valid,      "IID"),
]:
    df_renamed = df.rename(columns={id_col: "Patient_ID"})
    merged = target.merge(df_renamed, on="Patient_ID", how="inner")
    feature_cols = [c for c in df_renamed.columns if c != "Patient_ID"]

    for col in feature_cols:
        sub = merged[["Mean_T1_septum", col]].dropna()
        if len(sub) < 10:
            continue
        r, p = pearsonr(sub["Mean_T1_septum"], sub[col])
        results.append({"source": df_name, "feature": col, "pearson_r": r, "p_value": p})

results_df = pd.DataFrame(results).sort_values("p_value")
results_df["significant"] = results_df["p_value"] < 0.05

print(f"Total comparisons: {len(results_df)}")
print(f"Significant (p<0.05): {results_df['significant'].sum()}")
print("\nTop 20 by |r|:")
print(results_df.reindex(results_df["pearson_r"].abs().sort_values(ascending=False).index).head(20).to_string(index=False))

In [ ]:
print(phenotypes_valid[["id", "Mean_T1_septum"]].dropna().shape[0])

In [ ]:
# Load the datasets
ischemic_df = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium/ischemic_T1_phenotypes_imputed.no_outliers.residuals.qnorm.txt", sep="\t")
nonischemic_df = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium/nonischemic_T1_phenotypes_imputed.no_outliers.residuals.qnorm.txt", sep="\t")
normal_df = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium/normal_T1_phenotypes_imputed.no_outliers.residuals.qnorm.txt", sep="\t")

# Concatenate the datasets
combined_df = pd.concat([ischemic_df, nonischemic_df, normal_df], ignore_index=True)

# Drop duplicates based on the 'FID' column, keeping the first occurrence
combined_df = combined_df.drop_duplicates(subset='FID', keep='first')

# Display or save the combined DataFrame
combined_df.to_csv("./data/shriya/SHMOLLI-output-unet-myocardium/combined_T1_phenotypes_imputed.no_outliers.residuals.qnorm", sep="\t", index=False)
combined_df
